# Bodhan Tool Server and Task Agent — Walkthrough

This guided walkthrough demonstrates:
1. Inspecting media files with zero model cost
2. Running the 7 Bodhan Indic MCP tools
3. Orchestrating a multi-step language task using the autonomous Task Agent
4. Reviewing the execution trace, budget, and verification reports

## 1. Setup & Environment
Ensure you have configured your Bodhan API keys in `.env`.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
sys.path.insert(0, str(Path.cwd() / "src"))

print("Configured Keys:")
for k in ["BODHAN_API_KEY", "BODHAN_TRANSCRIBE_KEY", "BODHAN_TRANSLATE_KEY", "BODHAN_SPEAK_KEY", "BODHAN_OCR_KEY"]:
    val = os.environ.get(k, "")
    status = "[SET]" if val else "[NOT SET]"
    print(f"  {k}: {status}")

## 2. FastMCP Tool Server
Initialize the FastMCP tool server and inspect all registered tools.

In [ ]:
from bodhan_mcp.server import create_server

server = create_server()
tools = await server.list_tools()
print(f"Registered {len(tools)} MCP Tools:")
for t in tools:
    print(f"- {t.name}: {t.description[:80]}...")

## 3. Tool Invocation Example: Inspect Media & Detect Language

In [ ]:
import json

# Inspect sample audio file
sample_wav = "sample_data/hindi_sample.wav"
res = await server.call_tool("inspect_media", {"file_path": sample_wav})
content = res[0].text if isinstance(res, list) else str(res)
print("Inspect Result:", json.dumps(json.loads(content), indent=2))

# Free language detection via Unicode script analysis
res_lang = await server.call_tool("detect_language", {"text": "नमस्ते दुनिया, आप कैसे हैं?"})
content_lang = res_lang[0].text if isinstance(res_lang, list) else str(res_lang)
print("Language Detection Result:", json.dumps(json.loads(content_lang), indent=2))

## 4. Run the Task Agent CLI
The task agent handles multi-step tasks end to end with verification and cost control.

In [ ]:
from bodhan_agent.loop import run_task

out_dir = Path("./outputs/walkthrough_run")

await run_task(
    task="Translate this sample note into Hindi and preserve numbers",
    files=Path("sample_data/sample_note.txt"),
    out_dir=out_dir,
    max_steps=10,
    max_cost_inr=5.0,
)

print("Generated Files:")
for f in out_dir.iterdir():
    print(f"- {f.name}")

## 5. Review Execution Summary (result.md)

In [ ]:
result_md = out_dir / "result.md"
if result_md.exists():
    print(result_md.read_text(encoding="utf-8"))